
# 00 — Why This Project Exists: A Physics Detour Before Any Machine Learning

**Goal of this notebook:** build the physics intuition from *zero*, with runnable
demos, before we touch a single neural network. Everything downstream in this
project — Methods A/B/C, the harmonic oscillator benchmark, even the NUTS
integrator comparison — is a consequence of the ideas in this notebook.

### The problem this project is trying to solve

Every neural network has two kinds of numbers:
- **Weights** — the millions of numbers the network learns *during* training.
- **Hyperparameters** — learning rate, dropout, weight decay, etc., set *before*
  training starts.

Weights are tuned automatically, thousands of times a second, by gradient
descent, *while the network trains*. Hyperparameters are tuned by a totally
separate, expensive outer loop: pick a config, train a whole model, check how
good it was, throw it away, try again.

**This project asks: what if hyperparameters didn't need a separate loop at
all — what if they could evolve *alongside* the weights, using the same kind
of physics that already makes training stable?**

That "same kind of physics" is Hamiltonian mechanics. Let's build it up.



## 1. A ball rolling in a bowl

Imagine a ball rolling in a bowl-shaped valley. At any instant it has:
- a **position** $q$ (where it is)
- a **momentum** $p$ (how fast, which direction)

Physicists call the pair $(q, p)$ **phase space**. The ball's total energy —
kinetic + potential — is called the **Hamiltonian** $H(q, p)$. For a simple
spring/pendulum system (a *harmonic oscillator*):

$$H(q, p) = \underbrace{\frac{p^2}{2m}}_{\text{kinetic}} + \underbrace{\frac{1}{2}kq^2}_{\text{potential}}$$

The remarkable fact: if you let the ball evolve under the *true* laws of
physics, $H$ never changes. Energy just trades between kinetic and potential
form — it's **conserved**.

### Why should a machine learning person care?

Training a neural network with gradient descent is like a ball rolling down a
hill (the loss landscape) — but with *no momentum concept* and *no energy
conservation*. That makes it prone to getting stuck, oscillating, or taking
badly-scaled steps. Hamiltonian Monte Carlo (HMC) fixes this by literally
simulating a physical ball rolling through the loss landscape, using a
momentum variable — but only if you simulate the physics *correctly*.

Let's see what "correctly" means, with numbers.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# A harmonic oscillator: H(q,p) = p^2/(2m) + 0.5*k*q^2
k, m = 1.0, 1.0
q0, p0 = 1.0, 0.0   # start: ball pulled out to q=1, at rest
dt, n_steps = 0.1, 200

def energy(q, p):
    return 0.5 * p**2 / m + 0.5 * k * q**2

# --- Method 1: plain (forward) Euler integration ---
# The "naive" way: just take a small step using the current slope.
def euler_step(q, p, dt):
    dq = p / m
    dp = -k * q
    return q + dt * dq, p + dt * dp

# --- Method 2: Leapfrog (symplectic) integration ---
# The trick: update momentum by a HALF step, then position by a FULL step,
# using that half-updated momentum, then momentum by another half step.
def leapfrog_step(q, p, dt):
    p_half = p - 0.5 * dt * k * q
    q_new  = q + dt * p_half / m
    p_new  = p_half - 0.5 * dt * k * q_new
    return q_new, p_new

def simulate(step_fn, q0, p0, dt, n_steps):
    q, p = q0, p0
    energies = [energy(q, p)]
    for _ in range(n_steps):
        q, p = step_fn(q, p, dt)
        energies.append(energy(q, p))
    return energies

euler_energies    = simulate(euler_step, q0, p0, dt, n_steps)
leapfrog_energies = simulate(leapfrog_step, q0, p0, dt, n_steps)

print(f"True energy (constant, by physics):     {energy(q0,p0):.4f}")
print(f"Euler:    starts at {euler_energies[0]:.4f}, ends at {euler_energies[-1]:.4f}  <- drifted a LOT")
print(f"Leapfrog: starts at {leapfrog_energies[0]:.4f}, ends at {leapfrog_energies[-1]:.4f}  <- stayed almost exact")

plt.figure(figsize=(8,4.5))
plt.plot(euler_energies, label="Euler integration (naive)", color="#C44E52")
plt.plot(leapfrog_energies, label="Leapfrog integration (symplectic)", color="#4C72B0")
plt.axhline(energy(q0,p0), color="gray", linestyle="--", linewidth=1, label="True constant energy")
plt.xlabel("Time step")
plt.ylabel("Total energy H(q,p)")
plt.title("Energy conservation: naive integration vs. symplectic (leapfrog) integration")
plt.legend()
plt.tight_layout()
plt.show()



### What just happened, and why it's the whole reason this project exists

Plain Euler integration's energy **grows over 7x** across the simulation —
that's a numerical artifact, not real physics; the simulated ball ends up with
way more energy than it started with, for no physical reason. Leapfrog's
energy stays essentially flat — it "shadows" the true conserved quantity
almost exactly, even though it's still just an approximation taking discrete
steps.

This isn't a coincidence — it's a mathematical property called being
**symplectic**: leapfrog exactly preserves a *slightly different* Hamiltonian
that stays extremely close to the true one, no matter how long you run it.
Euler has no such guarantee, and its errors compound over time.

**Why this matters for training neural networks:** Hamiltonian Monte Carlo
(HMC) uses exactly this leapfrog trick to propose new parameter values that
explore a loss landscape efficiently, without the sampler's own bookkeeping
"leaking" energy in a way that would make its statistics wrong. Methods A and
C in this project use leapfrog for exactly this reason.

**The big leap this project takes:** if leapfrog can conserve energy for a
position/momentum pair $(q, p)$ representing physical particles, why not let
$q$ represent *both the network's weights and its hyperparameters at once*?
That's the "augmented Hamiltonian" idea explored in the next notebook.



## 2. From one ball to a whole neural network

Instead of one $(q, p)$ pair, imagine:
- $\theta$ = all the network's weights (thousands to millions of numbers)
- $\lambda$ = the hyperparameters (learning rate, dropout, weight decay, ...)
- $p_\theta, p_\lambda$ = a momentum variable for *each*

The **augmented Hamiltonian** used throughout this project is:

$$H(\theta, \lambda, p_\theta, p_\lambda) = \underbrace{\frac{\|p_\theta\|^2}{2m_\theta} + \frac{\|p_\lambda\|^2}{2m_\lambda}}_{\text{kinetic energy of weights + hyperparameters}} + \underbrace{\mathcal{L}(\theta, \lambda)}_{\text{the training loss, as potential energy}}$$

The training loss $\mathcal{L}$ plays the role of the "potential energy" (the
bowl shape) — low loss is a low, stable valley; high loss is a steep hill.
Leapfrog integration lets weights *and* hyperparameters roll through this
joint landscape together, in one physically consistent trajectory, instead of
hyperparameters being tuned by a completely separate outer loop.

That's the whole idea. Everything else in this project is either **building**
this (Methods A/B/C), **testing** it honestly (the benchmarks and statistics),
or **probing its limits** (the real-world showcase and the NUTS comparison).

**Next notebook (`01`):** we load the actual project code and watch this joint
system evolve on a real neural network, live.
